## Selección del modelo final

El modelo `bagging_rf_model.pkl` fue seleccionado como artefacto final del Sprint 4 porque corresponde a un ensamble basado en Random Forest, alineado con el PB-14 de técnicas avanzadas.

Como parte del rol Experiment Tracker, el modelo fue copiado y persistido como:

`models/final_model.pkl`

Luego se validó que el modelo final puede cargarse correctamente con `joblib` y generar predicciones sobre una muestra del conjunto de prueba.

Además, el resultado final fue registrado en:

`models/experiments_log.csv`

con sus métricas, ruta del modelo, parámetros relevantes y marca de modelo seleccionado.

In [4]:
import joblib
from pathlib import Path

MODELS_DIR = Path("models") if Path("models").exists() else Path("../models")

bagging_path = MODELS_DIR / "bagging_rf_model.pkl"
# Asumimos que el archivo del umbral sigue existiendo en la carpeta
threshold_path = MODELS_DIR / "bagging_rf_threshold.pkl" 
final_path = MODELS_DIR / "final_model.pkl"

print("Existe bagging_rf_model.pkl:", bagging_path.exists())

# 1. Cargamos el modelo y el umbral específico
model_objeto = joblib.load(bagging_path)
umbral_objeto = joblib.load(threshold_path)

# 2. Creamos un contenedor (diccionario) con ambos elementos
artefacto_final = {
    "model": model_objeto,
    "threshold": umbral_objeto
}

# 3. Guardamos el contenedor unificado usando joblib
joblib.dump(artefacto_final, final_path)

print("¡Artefacto final unificado con su umbral específico!")
print("Guardado en:", final_path)

Existe bagging_rf_model.pkl: True
¡Artefacto final unificado con su umbral específico!
Guardado en: ..\models\final_model.pkl


In [6]:
import pandas as pd
import joblib
from pathlib import Path

# 1. Configuración de rutas
MODELS_DIR = Path("models") if Path("models").exists() else Path("../models")
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

# 2. Carga de datos de prueba
test_df = pd.read_csv(DATA_DIR / "processed" / "test_original.csv")
X_test = test_df.drop("y", axis=1)
y_test = test_df["y"]

# 3. Carga del artefacto unificado (Diccionario)
artefacto = joblib.load(MODELS_DIR / "final_model.pkl")

# 4. Desempaquetamos el modelo puro y tu umbral específico
modelo_puro = artefacto["model"]
umbral_optimo = artefacto["threshold"]

# 5. Tomamos las primeras 5 muestras y las convertimos a NumPy para evitar el error de columnas
X_test_sample = X_test.iloc[:5].to_numpy()

# 6. REEMPLAZO DE PREDICCIÓN: Usamos predict_proba + el umbral óptimo
probabilidades = modelo_puro.predict_proba(X_test_sample)[:, 1]
preds_con_umbral = (probabilidades >= umbral_optimo).astype(int)

# 7. Impresión de resultados
print(f"Umbral óptimo aplicado: {umbral_optimo:.4f}")
print("Predicciones del modelo final (con umbral ajustado):")
print(preds_con_umbral)

c:\Users\user\.conda\envs\environment\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but BaggingClassifier was fitted with feature names
  warnings.warn(


Umbral óptimo aplicado: 0.2038
Predicciones del modelo final (con umbral ajustado):
[0 0 0 0 0]


In [8]:
import sys
import os
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix, classification_report

sys.path.append(os.path.abspath(".."))

from src.tuning import log_sprint4_result # Solo importamos el logger

MODELS_DIR = Path("models") if Path("models").exists() else Path("../models")
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

test_df = pd.read_csv(DATA_DIR / "processed" / "test_original.csv")

X_test = test_df.drop("y", axis=1)
y_test = test_df["y"]

# 1. Cargamos el artefacto completo
artefacto = joblib.load(MODELS_DIR / "final_model.pkl")
modelo_puro = artefacto["model"]
umbral_optimo = artefacto["threshold"]

# 2. CALCULO MANUAL CON EL UMBRAL ÓPTIMO
# Convertimos a NumPy para evitar el error original de feature_names_in_
X_test_np = X_test.to_numpy() if hasattr(X_test, 'columns') else X_test

# Obtenemos probabilidades de la clase positiva
probabilidades = modelo_puro.predict_proba(X_test_np)[:, 1]
# Aplicamos tu umbral específico
preds_con_umbral = (probabilidades >= umbral_optimo).astype(int)

# 3. Construimos el diccionario de métricas con el formato que espera tu logger
precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds_con_umbral, average='binary')
auc_roc = roc_auc_score(y_test, probabilidades)

metrics = {
    "f1": precision,         # Revisa si tu JSON/CSV usa minúsculas (f1, recall, precision, auc_roc)
    "recall": recall,       # Si tu logger usa mayúsculas (F1, Recall), cámbialas aquí a la izquierda
    "precision": f1,
    "auc_roc": auc_roc
}

# Generamos reporte y matriz por si los necesitas visualizar en el notebook
report = classification_report(y_test, preds_con_umbral)
cm = confusion_matrix(y_test, preds_con_umbral)

# 4. Registramos el experimento de forma normal con las métricas corregidas
result = log_sprint4_result(
    model="Bagging Random Forest",
    experiment_type="Final Model Validation",
    metrics=metrics,
    params={
        "source_model": "bagging_rf_model.pkl",
        "final_artifact": "final_model.pkl",
        "applied_threshold": umbral_optimo
    },
    model_path=str(MODELS_DIR / "final_model.pkl"),
    stage="PB-15 Final Validation",
    selected=True,
    notes=f"Modelo final con umbral optimizado ({umbral_optimo:.4f}) validado en test set.",
    log_path=str(MODELS_DIR / "experiments_log.csv")
)

print("Métricas finales logradas con el umbral óptimo (Calculadas Manualmente):")
print(metrics)

print("\nReporte de Clasificación Completo:")
print(report)

c:\Users\user\.conda\envs\environment\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but BaggingClassifier was fitted with feature names
  warnings.warn(


Métricas finales logradas con el umbral óptimo (Calculadas Manualmente):
{'f1': 0.23481781376518218, 'recall': 0.7508090614886731, 'precision': 0.35774865073245954, 'auc_roc': 0.7981894827099785}

Reporte de Clasificación Completo:
              precision    recall  f1-score   support

           0       0.96      0.69      0.80      7307
           1       0.23      0.75      0.36       927

    accuracy                           0.70      8234
   macro avg       0.60      0.72      0.58      8234
weighted avg       0.87      0.70      0.75      8234

